In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, get_cosine_schedule_with_warmup
import torch
from TimexNormUtils import TemporalDataset, compute_metrics, collator

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EPOCHS = 15
LR = 5e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 8
MAX_NEW_TOK  = 64

cleandata_path = "D:\\GeoTKG\\cleandata\\normalise\\"
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collator(b, tokenizer))
eval_loader = torch.utils.data.DataLoader(eval, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collator(b, tokenizer))

steps_per_epoch = len(train_loader)
eval_steps_per_epoch = len(eval_loader)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(enabled=True)
sched = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.01*steps_per_epoch*NUM_EPOCHS), num_training_steps=steps_per_epoch*NUM_EPOCHS)

In [ ]:
tokenizer.add_special_tokens({"additional_special_tokens": ["DCT:", "TYPE:", "TEXT:", "SPAN:"]})
model.resize_token_embeddings(len(tokenizer))

In [ ]:
history = {"loss": [], "eval_loss":[], "acc":[], "relax_acc":[]}
model.to(device)
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        with torch.amp.autocast(enabled=True):
            outputs = model(**batch)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0.0
    decoded_preds, decoded_labels = [], []

    with torch.no_grad():
        for batch in eval_loader:
            # keep a copy of labels for loss and for decoding ground truth
            labels_copy = batch["labels"].clone()
            batch = {k: v.to(device) for k, v in batch.items()}

            # loss
            out = model(**batch)
            eval_loss += out.loss.item()

            # predictions
            gen_ids = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_new_tokens=MAX_NEW_TOK,
                num_beams=4,
                early_stopping=True
            )
            decoded_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))

            # decode ground truth labels (replace -100 → pad for decoding)
            labels_copy[labels_copy == -100] = tokenizer.pad_token_id
            decoded_labels.extend(tokenizer.batch_decode(labels_copy, skip_special_tokens=True))

    avg_eval_loss = eval_loss / max(1, len(eval_loader))
    metrics = compute_metrics(decoded_labels, decoded_preds)

    history["loss"].append(total_loss/steps_per_epoch)
    history["eval_loss"].append(avg_eval_loss)
    history["acc"].append(metrics["accuracy strict"])
    history["relax_acc"].append(metrics["accuracy relaxed"])
    print(f"EPOCH: {epoch+1} LOSS: {total_loss/steps_per_epoch:.4f} ACC: {metrics['accuracy strict']} RACC: {metrics['accuracy relaxed']}")
    if (epoch+1)%5==0:
        torch.save({'model_state_dict': model.state_dict()}, f"results/norm_model/time_norm_epoch{epoch+1}.pt")
        

In [ ]:
# del batch, outputs, loss, model
# torch.cuda.empty_cache()